# Lyra Operator Analytics Notebook

**Purpose:** Systematic interrogation of the Lyra dataset by the operator. Not a product feature. Not to be shipped to users.

**When to use:** Day 10 / 30 / 60 / 90 milestones during the trusted-user monitoring window (Phase 5). See `docs/operator_interrogation_checklist.md` for the question list that drives each session, and `docs/operator_findings_log.md` for the template to record what each pass surfaced.

**Workflow:**
1. Run Cell A to connect to the DB.
2. Run Cell B to load the primary joined dataframe.
3. Use helper cells (C–G) from the question-cell templates below. Each Day-10 question has a starter cell; duplicate, edit, rerun.
4. Each finding that warrants action (new dogfood item, new VT, noted-not-actionable) goes in `docs/operator_findings_log.md`.

**IMPORTANT — live data:** Save a working copy as `operator_analytics.local.ipynb` (gitignored) before running against real user data. The committed file is a clean template with no outputs.

**DB access:** See `notebooks/README.md` — SQLite lives inside the Docker named volume `lyra_data`, not bind-mounted to the host. Copy it out with `docker cp` or run this notebook inside the backend container.

---
## Cell A — Database connection + session setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sqlalchemy import create_engine, text

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 220)

# Default: host-side copy exported from the Docker volume.
# Override with env var LYRA_DB_PATH when running inside the backend container.
DB_PATH = os.environ.get(
    "LYRA_DB_PATH",
    str(Path.cwd().parent / "backend" / "app" / "lyra.db"),
)
if not Path(DB_PATH).exists():
    raise FileNotFoundError(
        f"DB not found at {DB_PATH}. See notebooks/README.md for export steps."
    )

engine = create_engine(f"sqlite:///{DB_PATH}")
print(f"Connected to {DB_PATH}")
with engine.connect() as conn:
    tables = conn.execute(text("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")).fetchall()
print("Tables:", [t[0] for t in tables])

---
## Cell B — Core dataframe loader

`load_sessions(cohort, date_range)` returns a pandas DataFrame joining `task`, `stopwatch_session`, and `user`. Derived columns (`duration_delta_minutes`, `discrepancy_score`, `signed_discrepancy`) are computed here in Python because the backend defines them as SQLAlchemy hybrid properties, not stored columns.

**Defaults:**
- `cohort="operator"` → `user.is_operator = 1`
- `cohort="trusted"` → `user.is_operator = 0` (everyone else)
- `cohort="all"` → no user filter
- `voided_at IS NULL` is always applied (per voided_at guard).

In [ ]:
def load_sessions(cohort: str = "operator", date_range: tuple | None = None) -> pd.DataFrame:
    """Load joined task × stopwatch_session × user dataframe.

    Args:
        cohort: 'operator', 'trusted', or 'all'.
        date_range: (start_iso, end_iso) filtering on planned_start_utc. None = no bound.

    Returns:
        DataFrame with one row per task, stopwatch session columns prefixed `ss_`.
        If a task has multiple sessions they are aggregated (first start, last end,
        summed paused minutes, mean completion pct). For per-session granularity,
        pass `explode_sessions=True` manually via a custom query below.
    """
    where = ["t.voided_at IS NULL"]
    params: dict = {}
    if cohort == "operator":
        where.append("u.is_operator = 1")
    elif cohort == "trusted":
        where.append("(u.is_operator = 0 OR u.is_operator IS NULL)")
    elif cohort != "all":
        raise ValueError(f"cohort must be 'operator'|'trusted'|'all', got {cohort!r}")
    if date_range is not None:
        where.append("t.planned_start_utc >= :start")
        where.append("t.planned_start_utc < :end")
        params["start"], params["end"] = date_range

    sql = f"""
    SELECT
      t.task_id, t.user_id, t.title, t.category, t.state, t.source,
      t.planned_start_utc, t.planned_end_utc, t.planned_duration_minutes,
      t.executed_start_utc, t.executed_end_utc, t.executed_duration_minutes,
      t.pre_task_readiness, t.post_task_reflection,
      t.initiation_status, t.initiation_delay_minutes,
      t.pause_count, t.reschedule_count,
      t.parent_task_id, t.interruption_type,
      t.replaced_by_task_id, t.replaces_task_id,
      t.session_index_in_day, t.unplanned_reason,
      t.voided_at, t.voided_reason,
      u.is_operator,
      ss.start_time_utc  AS ss_start_time_utc,
      ss.end_time_utc    AS ss_end_time_utc,
      ss.total_paused_minutes AS ss_total_paused_minutes,
      ss.pause_reason    AS ss_pause_reason,
      ss.pause_initiator AS ss_pause_initiator,
      ss.original_pre_task_readiness AS ss_original_pre_task_readiness,
      ss.task_completion_percentage  AS ss_task_completion_percentage
    FROM task t
    LEFT JOIN user u ON u.user_id = t.user_id
    LEFT JOIN stopwatch_session ss ON ss.task_id = t.task_id
    WHERE {' AND '.join(where)}
    ORDER BY t.planned_start_utc
    """
    df = pd.read_sql(text(sql), engine, params=params)

    for col in ("planned_start_utc", "planned_end_utc",
                "executed_start_utc", "executed_end_utc",
                "ss_start_time_utc", "ss_end_time_utc", "voided_at"):
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")

    # Hybrid-property equivalents (computed in Python — backend does the same).
    df["duration_delta_minutes"] = (
        df["planned_duration_minutes"] - df["executed_duration_minutes"]
    )
    df["discrepancy_score"] = (
        df["pre_task_readiness"] - df["post_task_reflection"]
    ).abs()
    df["signed_discrepancy"] = (
        df["post_task_reflection"] - df["pre_task_readiness"]
    )

    return df


df = load_sessions(cohort="operator")
print(f"Loaded {len(df):,} rows, {df['task_id'].nunique():,} distinct tasks.")
df.head()

---
## Cell C — Distribution plot helper

In [ ]:
def plot_distribution(df: pd.DataFrame, column: str, stratify_by: str | None = None,
                      bins: int = 30, clip_quantiles: tuple | None = (0.01, 0.99)):
    """Histogram of `column`, optionally split by `stratify_by`.

    clip_quantiles trims extreme tails so outliers do not flatten the rest of
    the distribution. Pass None to disable.
    """
    data = df[[column] + ([stratify_by] if stratify_by else [])].dropna(subset=[column])
    if clip_quantiles is not None:
        lo, hi = data[column].quantile(clip_quantiles)
        data = data[(data[column] >= lo) & (data[column] <= hi)]

    fig, ax = plt.subplots(figsize=(9, 4.5))
    if stratify_by:
        for key, grp in data.groupby(stratify_by, dropna=False):
            ax.hist(grp[column], bins=bins, alpha=0.45, label=f"{stratify_by}={key} (n={len(grp)})")
        ax.legend(loc="best", fontsize=9)
    else:
        ax.hist(data[column], bins=bins, alpha=0.75)

    ax.set_title(f"{column} distribution" + (f" by {stratify_by}" if stratify_by else ""))
    ax.set_xlabel(column)
    ax.set_ylabel("count")
    plt.tight_layout()
    plt.show()
    print(data[column].describe())

---
## Cell D — Stratified correlation helper

Computes Spearman ρ between `primary` and `target` within buckets of `confounder`. Used to rule out spurious correlations that are really driven by a third variable (e.g. a category effect masquerading as a readiness effect).

In [ ]:
def stratified_corr(df: pd.DataFrame, target: str, primary: str, confounder: str,
                    bucket_strategy: str = "quartile", min_n: int = 10) -> pd.DataFrame:
    """Spearman correlation between `primary` and `target` within buckets of `confounder`.

    bucket_strategy:
      'quartile' — 4 equal-frequency bins on numeric confounder
      'categorical' — use confounder values directly (for category/state/etc.)
    """
    sub = df[[target, primary, confounder]].dropna()
    if bucket_strategy == "quartile":
        sub = sub.copy()
        sub["__bucket"] = pd.qcut(sub[confounder], q=4, duplicates="drop")
    elif bucket_strategy == "categorical":
        sub["__bucket"] = sub[confounder].astype(str)
    else:
        raise ValueError(f"bucket_strategy must be 'quartile'|'categorical', got {bucket_strategy!r}")

    rows = []
    for bucket, grp in sub.groupby("__bucket", dropna=False):
        if len(grp) < min_n:
            rows.append({"bucket": str(bucket), "n": len(grp), "rho": np.nan, "p": np.nan,
                         "note": f"below min_n={min_n}"})
            continue
        rho, p = stats.spearmanr(grp[primary], grp[target])
        rows.append({"bucket": str(bucket), "n": len(grp),
                     "rho": round(rho, 3), "p": round(p, 4), "note": ""})

    # Unconditional baseline for comparison.
    rho_all, p_all = stats.spearmanr(sub[primary], sub[target])
    rows.append({"bucket": "<ALL>", "n": len(sub),
                 "rho": round(rho_all, 3), "p": round(p_all, 4), "note": "unstratified"})
    return pd.DataFrame(rows)

---
## Cell E — Time series helper

In [ ]:
def plot_over_time(df: pd.DataFrame, column: str, window: str = "D",
                   agg_func: str = "mean", time_col: str = "planned_start_utc"):
    """Rolling time-series of `column`.

    window: pandas offset alias ('D', 'W', '3D', ...)
    agg_func: 'mean' | 'median' | 'count' | 'sum' | 'std'
    """
    sub = df[[time_col, column]].dropna(subset=[time_col]).copy()
    sub = sub.set_index(time_col).sort_index()
    resampled = sub[column].resample(window).agg(agg_func)

    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(resampled.index, resampled.values, marker="o", markersize=3)
    ax.set_title(f"{agg_func}({column}) over time, window={window}")
    ax.set_xlabel(time_col)
    ax.set_ylabel(f"{agg_func}({column})")
    ax.axhline(resampled.mean(), color="grey", linestyle="--", alpha=0.5, label="overall mean")
    ax.legend()
    plt.tight_layout()
    plt.show()
    return resampled

---
## Cell F — Category × time-of-day heatmap

In [ ]:
def category_time_heatmap(df: pd.DataFrame, metric: str,
                          time_col: str = "planned_start_utc",
                          agg_func: str = "mean", min_cell_n: int = 3):
    """Category (rows) × hour-of-day (cols) heatmap of `metric`.

    Cells with fewer than `min_cell_n` observations are masked — averages
    computed on tiny samples mislead.
    """
    sub = df[[time_col, "category", metric]].dropna()
    sub = sub.copy()
    sub["hour"] = pd.to_datetime(sub[time_col], utc=True).dt.hour
    pivot = sub.pivot_table(index="category", columns="hour",
                            values=metric, aggfunc=agg_func)
    counts = sub.pivot_table(index="category", columns="hour",
                             values=metric, aggfunc="count")
    masked = pivot.where(counts >= min_cell_n)

    fig, ax = plt.subplots(figsize=(12, max(3, 0.45 * len(pivot))))
    sns.heatmap(masked, annot=True, fmt=".1f", cmap="RdBu_r", center=0, ax=ax,
                cbar_kws={"label": f"{agg_func}({metric})"})
    ax.set_title(f"category × hour — {agg_func}({metric}), cells <{min_cell_n} hidden")
    plt.tight_layout()
    plt.show()
    return masked, counts

---
## Cell G — Cascade chain visualizer

Plots `session_index_in_day` distributions and the per-day chain: each horizontal row is a day, markers show tasks by position-in-day with delta colouring. Interruption/parent links are annotated with arrows where `parent_task_id` is set. (Formal interruption-chain visualization ships in Phase 6 — this is the exploratory version.)

In [ ]:
def plot_cascade_chains(df: pd.DataFrame, date_range: tuple | None = None,
                        metric: str = "duration_delta_minutes"):
    """Cascade visualization: task position in day vs chosen metric, per day."""
    sub = df.dropna(subset=["session_index_in_day", "planned_start_utc", metric]).copy()
    sub["planned_date"] = sub["planned_start_utc"].dt.date
    if date_range is not None:
        start, end = pd.to_datetime(date_range[0]).date(), pd.to_datetime(date_range[1]).date()
        sub = sub[(sub["planned_date"] >= start) & (sub["planned_date"] < end)]

    if sub.empty:
        print("No rows after filtering — cascade plot skipped.")
        return sub

    days = sorted(sub["planned_date"].unique())
    fig, ax = plt.subplots(figsize=(11, max(3, 0.35 * len(days))))
    vmax = sub[metric].abs().quantile(0.95) or 1
    for y, day in enumerate(days):
        day_rows = sub[sub["planned_date"] == day].sort_values("session_index_in_day")
        ax.scatter(day_rows["session_index_in_day"], [y] * len(day_rows),
                   c=day_rows[metric], cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                   s=70, edgecolor="k", linewidth=0.3)
        # parent→child arrows
        for _, row in day_rows[day_rows["parent_task_id"].notna()].iterrows():
            parent = day_rows[day_rows["task_id"] == row["parent_task_id"]]
            if not parent.empty:
                ax.annotate("", xy=(row["session_index_in_day"], y),
                            xytext=(parent.iloc[0]["session_index_in_day"], y),
                            arrowprops=dict(arrowstyle="->", color="grey", alpha=0.6))

    ax.set_yticks(range(len(days)))
    ax.set_yticklabels([str(d) for d in days])
    ax.set_xlabel("session_index_in_day")
    ax.set_title(f"cascade chains — colour = {metric}, arrows = parent_task_id")
    plt.tight_layout()
    plt.show()
    return sub

---
# Day 10 question cells

Each question below has a starter cell that runs with minimal modification. See `docs/operator_interrogation_checklist.md` for full prose and Day 30/60/90 questions. Log what you find in `docs/operator_findings_log.md`.

## Delta — is duration_delta_minutes systematically biased?

In [ ]:
# Q: central tendency and spread of duration_delta_minutes; which direction dominates?
plot_distribution(df, "duration_delta_minutes")
print("median:", df["duration_delta_minutes"].median())
print("mean:", df["duration_delta_minutes"].mean())
print("% overran (delta<0):", (df["duration_delta_minutes"] < 0).mean().round(3))

In [ ]:
# Q: is the delta different across categories?
plot_distribution(df, "duration_delta_minutes", stratify_by="category")
df.groupby("category")["duration_delta_minutes"].agg(["count", "median", "mean", "std"]).round(2)

## Discrepancy — does pre_task_readiness predict post-task reflection?

In [ ]:
# Q (H1): does signed_discrepancy correlate with duration_delta_minutes, stratified by category?
# Also logs unstratified baseline. Watch for sign flips — that is the confound signature.
stratified_corr(df, target="duration_delta_minutes", primary="signed_discrepancy",
                confounder="category", bucket_strategy="categorical")

In [ ]:
# Q (VT-12a): is pre_task_readiness correlated with planned_duration_minutes?
# If so, low-readiness days may have shorter plans — motivated underestimation.
rho, p = stats.spearmanr(
    df[["pre_task_readiness", "planned_duration_minutes"]].dropna().values.T[0],
    df[["pre_task_readiness", "planned_duration_minutes"]].dropna().values.T[1],
)
print(f"VT-12a Spearman: rho={rho:.3f}, p={p:.4f}, n={df[['pre_task_readiness','planned_duration_minutes']].dropna().shape[0]}")

In [ ]:
# Q (VT-12b): does variance of duration_delta_minutes at readiness=5 collapse?
# Saturated ceiling would compress the top-of-scale into a single bucket.
df.groupby("pre_task_readiness")["duration_delta_minutes"].agg(["count", "std", "mean", "median"]).round(2)

## Initiation delay — when does EXECUTING lag PLANNED start?

In [ ]:
# Q: distribution of initiation_delay_minutes; is there a time-of-day or category pattern?
plot_distribution(df, "initiation_delay_minutes")
category_time_heatmap(df, metric="initiation_delay_minutes")

## Unplanned rate — how much of execution is off-plan?

In [ ]:
# Q: share of tasks with unplanned_reason set (source='unplanned' or retroactive note).
df["is_unplanned"] = df["unplanned_reason"].notna() | (df["source"] == "unplanned")
print("overall unplanned rate:", df["is_unplanned"].mean().round(3))
plot_over_time(df.assign(is_unplanned=df["is_unplanned"].astype(int)),
               "is_unplanned", window="D", agg_func="mean")

## Cascade — do early-day overruns propagate?

In [ ]:
# Q: does session_index_in_day predict duration_delta_minutes?
# If later-in-day tasks run progressively longer over plan, that is the cascade signature.
plot_cascade_chains(df)
df.groupby("session_index_in_day")["duration_delta_minutes"].agg(["count", "median", "mean"]).round(2)

## Data quality — is anything suspiciously missing or uniform?

In [ ]:
# Q: null-rate per research-relevant field. Any field at 100% populated with a default
# value (e.g. readiness always = 3) is a hardcoded-default contamination flag — see
# docs/do_not_add.md §Hardcoded default values.
research_cols = [
    "pre_task_readiness", "post_task_reflection",
    "initiation_delay_minutes", "pause_count",
    "ss_pause_reason", "ss_pause_initiator", "ss_task_completion_percentage",
    "unplanned_reason", "category", "reschedule_count",
]
quality = pd.DataFrame({
    "null_rate": df[research_cols].isna().mean().round(3),
    "n_unique":  df[research_cols].nunique(),
    "mode":      df[research_cols].mode(dropna=False).iloc[0],
    "mode_share":[
        (df[c] == df[c].mode(dropna=False).iloc[0]).mean().round(3)
        for c in research_cols
    ],
})
quality

---
# VT-17 — Pause-prediction integrity suite

**Pre-registered in MANIFESTO §VT-17.** Run at the end of each user's 7-day acceptance window. Thresholds below are frozen at feature launch; no post-hoc redefinition.

Kill-criterion dependency: VT-17a/b/c findings can independently trigger a per-user feature kill **even if acceptance rate is within range**. Acceptance rate alone is insufficient — a well-accepted suggestion can still be anchoring rather than helping.

**Thresholds (all require p < 0.05):**
- VT-17a — anchor drift: Spearman ρ ≤ −0.40, n ≥ 20 matched firings
- VT-17b — induced pauses: pause-rate increase ≥ 50 %
- VT-17c — prompted split: median first-pause shift ≥ 5 min toward prediction

---
## Cell VT-load — pause_event + pause_prediction_log loader

Joins each firing to its matching pause_event using the same rule the reconcile job applies: a pause_event whose `paused_at_utc` falls in `[fired_at, predicted_at + ACCEPTANCE_WINDOW_MINUTES]`. That gives one row per firing with both predicted and actual pause times present for the accepted subset. Non-accepted firings have `actual_pause_at = NaT`.

In [ ]:
ACCEPTANCE_WINDOW_MINUTES = 5  # mirrors backend/app/workers/jobs/reconcile_responses.py


def load_pause_predictions(cohort: str = "all") -> pd.DataFrame:
    where = []
    if cohort == "operator":
        where.append("u.is_operator = 1")
    elif cohort == "trusted":
        where.append("(u.is_operator = 0 OR u.is_operator IS NULL)")
    elif cohort != "all":
        raise ValueError(f"cohort must be 'operator'|'trusted'|'all', got {cohort!r}")
    clause = f"WHERE {' AND '.join(where)}" if where else ""
    sql = f"""
    SELECT ppl.firing_id, ppl.user_id, ppl.fired_at, ppl.predicted_at,
           ppl.mechanism, ppl.confidence, ppl.lead_minutes, ppl.sample_size,
           ppl.active_task_id, ppl.user_response, ppl.response_at,
           ppl.parent_firing_id, u.is_operator
      FROM pause_prediction_log ppl
      LEFT JOIN user u ON u.user_id = ppl.user_id
      {clause}
      ORDER BY ppl.user_id, ppl.fired_at
    """
    df = pd.read_sql(text(sql), engine)
    for c in ("fired_at", "predicted_at", "response_at"):
        df[c] = pd.to_datetime(df[c], utc=True, errors="coerce")

    pe = pd.read_sql(text("SELECT user_id, paused_at_utc FROM pause_event"), engine)
    pe["paused_at_utc"] = pd.to_datetime(pe["paused_at_utc"], utc=True, errors="coerce")
    pe = pe.sort_values(["user_id", "paused_at_utc"])

    def match_row(row):
        if row.user_response != "pause_now":
            return pd.NaT
        window_end = row.predicted_at + pd.Timedelta(minutes=ACCEPTANCE_WINDOW_MINUTES)
        cands = pe[(pe.user_id == row.user_id)
                   & (pe.paused_at_utc >= row.fired_at)
                   & (pe.paused_at_utc <= window_end)]
        if cands.empty:
            return pd.NaT
        return cands.iloc[(cands.paused_at_utc - row.predicted_at).abs().argsort()].iloc[0].paused_at_utc

    df["actual_pause_at"] = df.apply(match_row, axis=1) if len(df) else pd.Series([], dtype="datetime64[ns, UTC]")
    df["gap_minutes"] = (df["actual_pause_at"] - df["predicted_at"]).dt.total_seconds() / 60.0
    return df


def load_pause_events(cohort: str = "all") -> pd.DataFrame:
    where = ["pe.paused_at_utc IS NOT NULL"]
    if cohort == "operator":
        where.append("u.is_operator = 1")
    elif cohort == "trusted":
        where.append("(u.is_operator = 0 OR u.is_operator IS NULL)")
    sql = f"""
    SELECT pe.pause_event_id, pe.session_id, pe.user_id,
           pe.paused_at_utc, pe.resumed_at_utc,
           pe.duration_minutes, pe.pause_reason, pe.pause_initiator,
           u.is_operator
      FROM pause_event pe
      LEFT JOIN user u ON u.user_id = pe.user_id
      WHERE {' AND '.join(where)}
      ORDER BY pe.user_id, pe.paused_at_utc
    """
    df = pd.read_sql(text(sql), engine)
    for c in ("paused_at_utc", "resumed_at_utc"):
        df[c] = pd.to_datetime(df[c], utc=True, errors="coerce")
    return df


predictions = load_pause_predictions(cohort="all")
pause_events = load_pause_events(cohort="all")
print(f"firings: {len(predictions):,}  pause_events: {len(pause_events):,}")
if len(predictions):
    print("by mechanism:", predictions["mechanism"].value_counts(dropna=False).to_dict())
    print("by user_response:", predictions["user_response"].value_counts(dropna=False).to_dict())

---
## Cell VT-17a — Anchor drift (per user)

For each user whose notification window is ≥ 7 days, compute Spearman ρ between day-index-within-window (0, 1, 2, …) and `|actual_pause_at − predicted_at|`. A negative ρ = the gap is shrinking over time → the most parsimonious explanation is that the suggestion is anchoring the user's pause timing rather than predicting it.

Trip condition: ρ ≤ −0.40, p < 0.05, n ≥ 20 matched firings per user.

In [ ]:
def vt17a_anchor_drift(df_predictions: pd.DataFrame, min_n: int = 20) -> pd.DataFrame:
    matched = df_predictions.dropna(subset=["actual_pause_at"]).copy()
    if matched.empty:
        print("No accepted firings yet — VT-17a deferred until pause_now responses accumulate.")
        return pd.DataFrame()
    matched["abs_gap_minutes"] = matched["gap_minutes"].abs()
    rows = []
    for user_id, grp in matched.groupby("user_id"):
        grp = grp.sort_values("fired_at").copy()
        window_days = (grp["fired_at"].max() - grp["fired_at"].min()).days
        grp["day_index"] = (grp["fired_at"] - grp["fired_at"].min()).dt.days
        n = len(grp)
        if n < min_n or window_days < 7:
            rows.append({"user_id": user_id, "n": n, "window_days": window_days,
                         "rho": np.nan, "p": np.nan, "trips_threshold": False,
                         "note": f"n={n} (<{min_n}) or window={window_days}d (<7)"})
            continue
        rho, p = stats.spearmanr(grp["day_index"], grp["abs_gap_minutes"])
        trips = bool(rho <= -0.40 and p < 0.05)
        rows.append({"user_id": user_id, "n": n, "window_days": window_days,
                     "rho": round(rho, 3), "p": round(p, 4),
                     "trips_threshold": trips,
                     "note": "ANCHORING DETECTED" if trips else ""})
    out = pd.DataFrame(rows)
    if not out.empty and out["trips_threshold"].any():
        tripped = out.loc[out.trips_threshold, "user_id"].tolist()
        print(f"⚠ VT-17a TRIPPED for user_id(s): {tripped}")
        print("  → Phase-6 bias_factor inputs for pause-timing categories must exclude")
        print("    predictions fired during the notification window for these users.")
        print("  → Log the finding in docs/operator_findings_log.md with the ρ/p/n values.")
    return out


vt17a_anchor_drift(predictions)

---
## Cell VT-17b — Induced pause rate (paired per user)

For each user: compute pauses-per-active-session in the pre-notification baseline (≥ 7 days of activity before the first firing) and in the notification window (from the first firing onward). Wilcoxon signed-rank paired across users.

Trip condition: pause-rate increase ≥ 50 %, p < 0.05. With fewer than 6 usable users the Wilcoxon is deferred — per-user pct_change is informative but not yet statistically distinguishable from noise.

In [ ]:
def vt17b_induced_pause_rate(df_predictions: pd.DataFrame,
                             df_events: pd.DataFrame,
                             min_baseline_days: int = 7) -> dict:
    if df_predictions.empty:
        return {"note": "no firings yet — VT-17b deferred"}

    sess = pd.read_sql(text(
        "SELECT ss.session_id, ss.task_id, ss.start_time_utc, t.user_id "
        "FROM stopwatch_session ss LEFT JOIN task t ON t.task_id = ss.task_id "
        "WHERE t.voided_at IS NULL"
    ), engine)
    sess["start_time_utc"] = pd.to_datetime(sess["start_time_utc"], utc=True, errors="coerce")

    first_fire = df_predictions.groupby("user_id")["fired_at"].min()

    rows = []
    for user_id, boundary in first_fire.items():
        user_sess = sess[sess.user_id == user_id]
        user_pauses = df_events[df_events.user_id == user_id]
        if user_sess.empty:
            continue
        baseline_sess = user_sess[user_sess.start_time_utc < boundary]
        baseline_days = (boundary - user_sess.start_time_utc.min()).days if not user_sess.empty else 0
        if baseline_days < min_baseline_days or baseline_sess.empty:
            rows.append({"user_id": user_id, "baseline_rate": np.nan,
                         "notif_rate": np.nan, "pct_change": np.nan,
                         "note": f"baseline={baseline_days}d (<{min_baseline_days}) or no sessions"})
            continue
        notif_sess = user_sess[user_sess.start_time_utc >= boundary]
        if notif_sess.empty:
            rows.append({"user_id": user_id, "baseline_rate": np.nan,
                         "notif_rate": np.nan, "pct_change": np.nan,
                         "note": "no notification-window sessions"})
            continue
        baseline_pauses = user_pauses[user_pauses.paused_at_utc < boundary]
        notif_pauses = user_pauses[user_pauses.paused_at_utc >= boundary]
        b_rate = len(baseline_pauses) / len(baseline_sess)
        n_rate = len(notif_pauses) / len(notif_sess)
        pct = (n_rate - b_rate) / b_rate * 100 if b_rate > 0 else np.nan
        rows.append({"user_id": user_id, "baseline_rate": round(b_rate, 3),
                     "notif_rate": round(n_rate, 3),
                     "pct_change": round(pct, 1) if pd.notna(pct) else np.nan,
                     "note": ""})
    per_user = pd.DataFrame(rows)
    print(per_user)

    usable = per_user.dropna(subset=["baseline_rate", "notif_rate"])
    if len(usable) < 6:
        return {"per_user": per_user, "wilcoxon": None,
                "note": f"n={len(usable)} users <6 — Wilcoxon deferred."}

    stat, p = stats.wilcoxon(usable.baseline_rate, usable.notif_rate)
    median_pct = usable.pct_change.median()
    trips = bool(median_pct >= 50 and p < 0.05)
    if trips:
        print(f"⚠ VT-17b TRIPPED: median pct_change={median_pct:.1f}% @ p={p:.4f}")
        print("  → Log in docs/operator_findings_log.md. Feature per-user kill candidate.")
    return {"per_user": per_user,
            "wilcoxon": {"stat": float(stat), "p": float(p),
                         "median_pct_change": float(median_pct),
                         "trips_threshold": trips}}


vt17b_induced_pause_rate(predictions, pause_events)

---
## Cell VT-17c — Natural vs prompted first-pause split

For each user-day in the notification window, split by whether a firing occurred **before** the first pause of that day. Compare the distribution of first-pause times (local minutes-from-midnight) between the prompted and natural subsets via Mann-Whitney U.

Trip condition: median first-pause in the prompted subset is ≥ 5 minutes earlier than in the natural subset (i.e. shifted toward the prediction), p < 0.05. Requires ≥ 5 observations in each subset.

In [ ]:
def vt17c_natural_vs_prompted(df_predictions: pd.DataFrame,
                              df_events: pd.DataFrame,
                              tz: str = "Africa/Cairo") -> dict:
    if df_predictions.empty or df_events.empty:
        return {"note": "insufficient data — predictions and/or pause_events are empty"}

    events = df_events.copy()
    events["local_paused"] = events["paused_at_utc"].dt.tz_convert(tz)
    events["local_date"] = events["local_paused"].dt.date
    first_pauses = (events.sort_values(["user_id", "local_paused"])
                          .groupby(["user_id", "local_date"], as_index=False)
                          .first())
    first_pauses["first_pause_minutes_from_midnight"] = (
        first_pauses["local_paused"].dt.hour * 60 + first_pauses["local_paused"].dt.minute
    )

    firings = df_predictions.copy()
    firings["local_fired"] = firings["fired_at"].dt.tz_convert(tz)
    firings["local_date"] = firings["local_fired"].dt.date

    # Only user-days on or after the user's first firing count.
    user_window_start = (df_predictions.groupby("user_id")["fired_at"].min()
                                       .dt.tz_convert(tz).dt.date)
    first_pauses = first_pauses.merge(
        user_window_start.rename("window_start"), on="user_id", how="left"
    )
    first_pauses = first_pauses.dropna(subset=["window_start"])
    first_pauses = first_pauses[first_pauses["local_date"] >= first_pauses["window_start"]]

    def prompted(row):
        fires = firings[(firings.user_id == row.user_id)
                        & (firings.local_date == row.local_date)
                        & (firings.local_fired <= row.local_paused)]
        return not fires.empty

    if first_pauses.empty:
        return {"note": "no user-days inside any notification window yet — VT-17c deferred"}

    first_pauses["prompted"] = first_pauses.apply(prompted, axis=1)
    prompted_set = first_pauses.loc[first_pauses.prompted, "first_pause_minutes_from_midnight"]
    natural_set = first_pauses.loc[~first_pauses.prompted, "first_pause_minutes_from_midnight"]
    if len(prompted_set) < 5 or len(natural_set) < 5:
        return {"prompted_n": len(prompted_set), "natural_n": len(natural_set),
                "note": "n<5 in one subset — Mann-Whitney deferred"}

    stat, p = stats.mannwhitneyu(prompted_set, natural_set, alternative="two-sided")
    median_shift = natural_set.median() - prompted_set.median()  # +ve = prompted earlier
    trips = bool(median_shift >= 5 and p < 0.05)
    out = {"prompted_n": len(prompted_set), "natural_n": len(natural_set),
           "prompted_median_min": round(float(prompted_set.median()), 1),
           "natural_median_min": round(float(natural_set.median()), 1),
           "median_shift_min": round(float(median_shift), 1),
           "u_stat": float(stat), "p": round(float(p), 4),
           "trips_threshold": trips}
    if trips:
        print(f"⚠ VT-17c TRIPPED: prompted first-pauses are {median_shift:.1f} min earlier "
              f"than natural (p={p:.4f}). Log in docs/operator_findings_log.md.")
    return out


vt17c_natural_vs_prompted(predictions, pause_events)